In [ ]:
# Configuracion de cluster Slurm (alineada con run.sh / run_parallel.sh)
SLURM_PARTITION = 'long'
SLURM_CPUS_PER_TASK = 8
SLURM_MEM = '32G'
SLURM_GRES_EXPERIMENT = 'shard:4'
SLURM_GRES_MASSIVE = 'shard:6'
CONDA_ENV = 'RFA2526pt'
PYTORCH_CUDA_ALLOC_CONF = 'expandable_segments:True'

import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = PYTORCH_CUDA_ALLOC_CONF

print('Partition:', SLURM_PARTITION)
print('CPUs:', SLURM_CPUS_PER_TASK, '| Mem:', SLURM_MEM)
print('GRES exp:', SLURM_GRES_EXPERIMENT, '| GRES masivo:', SLURM_GRES_MASSIVE)
print('Conda env:', CONDA_ENV)
print('PYTORCH_CUDA_ALLOC_CONF:', os.environ['PYTORCH_CUDA_ALLOC_CONF'])


# 07 - Experimento Sensorial Filter

Objetivo tecnico:
1. Filtrar outliers por modalidad sensorial (ET, EEG, HR) con regla IQR por muestra.
2. Construir estadisticas robustas por usuario y por feature.
3. Derivar caracteristicas por ventanas temporales (inicio/medio/final) alineadas con segmentos del texto Whisper.
4. Entrenar regresion logistica para ES, EN y ES+EN y exportar tres JSON.


In [ ]:
import json
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

DATA_JSON_PATH = Path('/home/alumno.upv.es/scheng1/EXIST 2026 Videos Dataset/training/EXIST2026_training.json')
PROJECT_ROOT = Path('/home/alumno.upv.es/scheng1/Master-IA-ALC/lab3/notebooks_shiyi')
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
DELIVERABLES_DIR = PROJECT_ROOT / 'entregables'
CACHE_NPZ = ARTIFACTS_DIR / 'sensorial_filtered_features.npz'

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
DELIVERABLES_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
with open(DATA_JSON_PATH, 'r', encoding='utf-8') as f:
    raw = json.load(f)

rows = []
for sid, obj in raw.items():
    rec = {'sample_id': str(sid)}
    rec.update(obj)
    rows.append(rec)

df = pd.DataFrame(rows)

def majority_task3_1(lbls):
    vals = [x for x in lbls if x in {'YES', 'NO'}]
    if not vals:
        return 'UNKNOWN'
    c = Counter(vals)
    if c['YES'] == c['NO']:
        return vals[0]
    return c.most_common(1)[0][0]

df['label'] = df['labels_task3_1'].apply(majority_task3_1)
df['y'] = df['label'].map({'NO': 0, 'YES': 1}).fillna(-1).astype(int)
df = df.sort_values('sample_id').reset_index(drop=True)
print('n=', len(df), '| langs=', df['lang'].value_counts().to_dict())


In [ ]:
def iqr_clip(values):
    arr = np.array([v for v in values if isinstance(v, (int, float, np.integer, np.floating)) and np.isfinite(v)], dtype=np.float32)
    if arr.size == 0:
        return arr
    q1, q3 = np.percentile(arr, [25, 75])
    iqr = q3 - q1
    if iqr == 0:
        return arr
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return np.clip(arr, lo, hi)


def text_window_weights(text, n_windows=3):
    toks = str(text).split()
    n = max(len(toks), 1)
    cuts = np.linspace(0, n, n_windows + 1, dtype=int)
    lengths = [cuts[i + 1] - cuts[i] for i in range(n_windows)]
    w = np.array(lengths, dtype=np.float32)
    w = w / (w.sum() + 1e-8)
    return w


def extract_sensorial_features(sample_sensorial, text):
    # Paso 1: recolectar por feature las observaciones multiusuario.
    per_feat = defaultdict(list)
    if isinstance(sample_sensorial, dict):
        modalities = sample_sensorial.get('modalities', {})
        for mod_name, mod_payload in modalities.items():
            by_user = mod_payload.get('by_user', {}) if isinstance(mod_payload, dict) else {}
            for _, feat_map in by_user.items():
                if not isinstance(feat_map, dict):
                    continue
                for feat_name, feat_val in feat_map.items():
                    if isinstance(feat_val, (int, float, np.integer, np.floating)) and np.isfinite(feat_val):
                        per_feat[f'{mod_name.lower()}__{feat_name}'].append(float(feat_val))

    # Paso 2: filtrar outliers y calcular estadisticas robustas.
    robust = {}
    for feat, vals in per_feat.items():
        vc = iqr_clip(vals)
        if vc.size == 0:
            continue
        robust[f'{feat}__mean'] = float(vc.mean())
        robust[f'{feat}__median'] = float(np.median(vc))
        robust[f'{feat}__std'] = float(vc.std())
        robust[f'{feat}__min'] = float(vc.min())
        robust[f'{feat}__max'] = float(vc.max())

    # Paso 3: ventanas temporales alineadas con Whisper.
    # Como no hay series crudas por timestamp en el JSON, usamos una aproximacion:
    # ventanas (inicio/medio/final) ponderadas por segmentos de transcripcion.
    w1, w2, w3 = text_window_weights(text, n_windows=3)
    out = {'window_w1': float(w1), 'window_w2': float(w2), 'window_w3': float(w3)}

    for feat in list(per_feat.keys()):
        fmin = robust.get(f'{feat}__min', robust.get(f'{feat}__mean', 0.0))
        fmid = robust.get(f'{feat}__mean', 0.0)
        fmax = robust.get(f'{feat}__max', robust.get(f'{feat}__mean', 0.0))
        fstd = robust.get(f'{feat}__std', 0.0)

        out[f'{feat}__win_inicio'] = float(fmin * w1)
        out[f'{feat}__win_medio'] = float(fmid * w2)
        out[f'{feat}__win_final'] = float(fmax * w3)
        out[f'{feat}__dispersion'] = float(fstd)

    out['sensor_user_count'] = float(len(sample_sensorial.get('users', [])) if isinstance(sample_sensorial, dict) else 0)
    return out


In [ ]:
if CACHE_NPZ.exists():
    cache = np.load(CACHE_NPZ, allow_pickle=True)
    X_sensor = cache['X_sensor']
    sid_cache = cache['sample_ids'].astype(str)
    assert list(sid_cache) == df['sample_id'].astype(str).tolist()
    print('Loaded cache:', CACHE_NPZ, '| shape=', X_sensor.shape)
else:
    feat_rows = []
    for row in df.itertuples(index=False):
        feats = extract_sensorial_features(row.sensorial, row.text)
        feats['sample_id'] = str(row.sample_id)
        feat_rows.append(feats)

    fdf = pd.DataFrame(feat_rows).sort_values('sample_id').reset_index(drop=True)
    sids = fdf['sample_id'].astype(str).values
    X_sensor = fdf.drop(columns=['sample_id']).fillna(0.0).to_numpy(np.float32)

    np.savez_compressed(CACHE_NPZ, X_sensor=X_sensor, sample_ids=sids)
    print('Saved cache:', CACHE_NPZ, '| shape=', X_sensor.shape)


In [ ]:
def train_and_export_three_lang_models(X, y, langs, sample_ids, exp_tag):
    configs = {
        'ES': ['es'],
        'EN': ['en'],
        'ES_EN': ['es', 'en'],
    }

    for cfg_name, cfg_langs in configs.items():
        m = np.isin(langs, cfg_langs) & (y >= 0)
        Xtr, ytr = X[m], y[m]

        clf = Pipeline([
            ('scaler', StandardScaler()),
            ('lr', LogisticRegression(max_iter=2500, class_weight='balanced', n_jobs=-1)),
        ])
        clf.fit(Xtr, ytr)

        pred = clf.predict(X)
        pred_labels = np.where(pred == 1, 'YES', 'NO')

        out = [
            {'test_case': 'EXIST2026', 'id': str(sid), 'value': str(lbl)}
            for sid, lbl in zip(sample_ids, pred_labels)
        ]
        out_path = DELIVERABLES_DIR / f'BeingChillingWeWillWin_{exp_tag}_{cfg_name}.json'
        with open(out_path, 'w', encoding='utf-8') as f:
            json.dump(out, f, ensure_ascii=False)
        print('Saved', out_path, '| rows=', len(out))


train_and_export_three_lang_models(
    X_sensor,
    df['y'].to_numpy(np.int64),
    df['lang'].astype(str).to_numpy(),
    df['sample_id'].astype(str).to_numpy(),
    exp_tag='07SensorialFilter_IQRWin',
)
